# UniverSeg Complexity Benchmark

This notebook benchmarks inference speed and memory usage for **Prompt U-Net (v330/v332 architecture)** vs **UniverSeg**.

**Settings evaluated:**
1. **Mega-Batch Mode**: Prompt U-Net uses a batch size large enough to fit all tiles and slices from the 3D volume into a single forward pass. UniverSeg processes slices sequentially (Batch=1). Evaluated across Small, Medium, and Large structures.
2. **Sequential Native**: Both models process 1 slice at a time (Batch=1). Evaluated ONLY on Small structures (128x128) to directly compare native forward-pass speed without any tiling overhead.

**Structures evaluated (40 2D slices each):**
- **Small (128x128)**
- **Medium (256x256)**
- **Large (512x512)**

*Note: This benchmark uses an isolated subprocess pattern. Each model runs in a separate process to guarantee that TensorFlow and PyTorch memory allocations do not conflict and that GPU memory is perfectly reset between evaluations.*


In [ ]:
import os
import sys
import json
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

SIZES = {
    "Small (128x128)": (128, 128),
    "Medium (256x256)": (256, 256),
    "Large (512x512)": (512, 512)
}

ENVIRONMENTS = [
    "Mega-Batch (P-UNet: All tiles, UniverSeg: Batch=1)", 
    "Sequential Native (Batch=1, Small Only)"
]
MODELS = ["Prompt U-Net", "UniverSeg"]


In [ ]:
worker_code = """import sys
import os
import gc
import time
import json
import numpy as np

# Setup paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

def clean_memory():
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
    if 'tensorflow' in sys.modules:
        import tensorflow as tf
        tf.keras.backend.clear_session()
        try:
            tf.config.experimental.reset_memory_stats('GPU:0')
        except:
            pass

def main():
    model_name = sys.argv[1]
    env = sys.argv[2]
    H = int(sys.argv[3])
    W = int(sys.argv[4])
    
    if "Mega-Batch" in env:
        punet_batch = 1000  # Large enough to fit 640 tiles (Large structure, 40 slices)
        universeg_batch = 1
    else:
        punet_batch = 1
        universeg_batch = 1
        
    NUM_SLICES = 40
    
    clean_memory()
    
    if model_name == "Prompt U-Net":
        import tensorflow as tf
        # Set TF memory growth
        gpus = tf.config.experimental.list_physical_devices('GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            
        from inference.predictor import PromptUNetPredictor
        punet_path = os.path.join(PROJECT_ROOT, "training", "p_unet_330.keras")
        punet_predictor = PromptUNetPredictor(punet_path)
        
        device_name = '/GPU:0'
        
        x = np.random.randn(NUM_SLICES, H, W).astype(np.float32)
        p = np.zeros((NUM_SLICES, H, W, 2), dtype=np.float32)
        
        y_start, y_end = int(H * 0.1), int(H * 0.9)
        x_start, x_end = int(W * 0.1), int(W * 0.9)
        p[:, y_start:y_end, x_start:x_end, 1] = 1.0 
        
        clean_memory()
        
        # Warmup pass
        with tf.device(device_name):
            _ = punet_predictor.predict(x, p, batch_size=punet_batch)
        
        clean_memory()
        
        # Timed run
        start_time = time.perf_counter()
        with tf.device(device_name):
            _ = punet_predictor.predict(x, p, batch_size=punet_batch)
        end_time = time.perf_counter()
        
        try:
            mem_info = tf.config.experimental.get_memory_info('GPU:0')
            mem_mb = mem_info['peak'] / (1024**2)
        except:
            mem_mb = 0.0
            
        return (end_time - start_time) * 1000, mem_mb

    elif model_name == "UniverSeg":
        import torch
        from evaluation.benchmark_models.UniverSeg.universeg import universeg
        
        device = torch.device('cuda:0')
        
        universeg_model = universeg(pretrained=False)
        universeg_model.eval()
        model = universeg_model.to(device)
        
        x = torch.randn(NUM_SLICES, 1, H, W)
        sup_x = torch.randn(NUM_SLICES, 16, 1, 128, 128)
        sup_y = torch.zeros(NUM_SLICES, 16, 1, 128, 128)
        
        clean_memory()
        
        # Warmup
        with torch.no_grad():
            x_128 = torch.nn.functional.interpolate(x[:1].to(device), size=(128, 128), mode='bilinear', align_corners=False)
            _ = model(x_128, sup_x[:1].to(device), sup_y[:1].to(device))
            torch.cuda.synchronize()
        
        clean_memory()
        
        # Timed run
        start_time = time.perf_counter()
        with torch.no_grad():
            for i in range(0, NUM_SLICES, universeg_batch):
                batch_x = x[i:i+universeg_batch].to(device)
                batch_sx = sup_x[i:i+universeg_batch].to(device)
                batch_sy = sup_y[i:i+universeg_batch].to(device)
                
                batch_x_128 = torch.nn.functional.interpolate(batch_x, size=(128, 128), mode='bilinear', align_corners=False)
                logits = model(batch_x_128, batch_sx, batch_sy)
                _ = torch.nn.functional.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
                
        torch.cuda.synchronize()
        end_time = time.perf_counter()
        
        mem_mb = torch.cuda.max_memory_allocated(device) / (1024**2) if device.type == 'cuda' else 0.0
        return (end_time - start_time) * 1000, mem_mb

if __name__ == "__main__":
    try:
        t, mem = main()
        print(json.dumps({"time": t, "memory": mem}))
    except Exception as e:
        print(json.dumps({"error": str(e)}))
"""

with open("benchmark_worker.py", "w", encoding="utf-8") as f:
    f.write(worker_code)
print("Worker script written to benchmark_worker.py")


In [ ]:
# Run Benchmarks
results = []

for env in ENVIRONMENTS:
    print(f"\n--- Evaluating on {env} ---")
    for size_name, (H, W) in SIZES.items():
        if "Small Only" in env and "Small" not in size_name:
            continue
            
        print(f"  Structure Size: {size_name}")
        
        for model in MODELS:
            # Execute worker process
            cmd = [sys.executable, "benchmark_worker.py", model, env, str(H), str(W)]
            proc = subprocess.run(cmd, capture_output=True, text=True)
            
            if proc.returncode != 0:
                print(f"    {model:<13} : Failed | {proc.stderr.strip()[:100]}")
                results.append({"Environment": env, "Size": size_name, "Model": model, "Time (ms)": np.nan, "Memory (MB)": np.nan})
                continue
                
            try:
                out_lines = [l for l in proc.stdout.split("\n") if l.strip().startswith("{") and "time" in l]
                if out_lines:
                    res = json.loads(out_lines[-1])
                    if "error" in res:
                        print(f"    {model:<13} : Error | {res['error'][:100]}")
                        t, mem = np.nan, np.nan
                    else:
                        t, mem = res["time"], res["memory"]
                else:
                    print(f"    {model:<13} : Parse Err | {proc.stdout.strip()[:100]}")
                    t, mem = np.nan, np.nan
            except Exception as e:
                print(f"    {model:<13} : Exception | {e}")
                t, mem = np.nan, np.nan
                
            results.append({"Environment": env, "Size": size_name, "Model": model, "Time (ms)": t, "Memory (MB)": mem})
            if np.isnan(t):
                pass # Error message already printed
            else:
                print(f"    {model:<13} : {t:.1f} ms, {mem:.1f} MB")

df = pd.DataFrame(results)
display(df)

# Save results for independent plotting
df.to_pickle("complexity_universeg_results.pkl")
print("Results saved to complexity_universeg_results.pkl")

# Cleanup
if os.path.exists("benchmark_worker.py"):
    os.remove("benchmark_worker.py")


In [ ]:
# Visualization

# Load results if they exist (allows running this cell independently)
if os.path.exists("complexity_universeg_results.pkl"):
    df = pd.read_pickle("complexity_universeg_results.pkl")

# Filter out OOM (NaN) for plotting
df_plot = df.dropna()

# Inference Time Plot
g_time = sns.catplot(
    data=df, kind="bar",
    x="Size", y="Time (ms)", hue="Model", col="Environment",
    palette=["#4c72b0", "#dd8452"],
    height=6, aspect=1.2, sharey=True
)
g_time.fig.suptitle("Volume Inference Time (40 slices) - Lower is Better", y=1.05, fontweight="bold")
for ax in g_time.axes.flat:
    ax.set_yscale("log")
    ax.set_ylabel("Time (ms) [Log Scale]")
    
    # Add annotations
    for p in ax.patches:
        height = p.get_height()
        if not np.isnan(height) and height > 0:
            ax.annotate(f'{height:.0f}', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, rotation=90, xytext=(0, 5),
                        textcoords='offset points')

g_time.savefig("complexity_universeg_time.pdf", format="pdf", bbox_inches="tight", dpi=300)
plt.show()

# Memory Plot
g_mem = sns.catplot(
    data=df, kind="bar",
    x="Size", y="Memory (MB)", hue="Model", col="Environment",
    palette=["#4c72b0", "#dd8452"],
    height=6, aspect=1.2, sharey=True
)
g_mem.fig.suptitle("Peak GPU Memory Usage (VRAM) - Lower is Better", y=1.05, fontweight="bold")
for ax in g_mem.axes.flat:
    ax.set_ylabel("Memory (MB)")
    
    # Add annotations
    for p in ax.patches:
        height = p.get_height()
        if not np.isnan(height) and height > 0:
            ax.annotate(f'{height:.0f}', (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, rotation=90, xytext=(0, 5),
                        textcoords='offset points')

g_mem.savefig("complexity_universeg_memory.pdf", format="pdf", bbox_inches="tight", dpi=300)
plt.show()
